# mt_evaluation_questions — 25 Evaluation Questions

Defines `EVALUATION_QUESTIONS` (Q1–Q25) and `_EXPECTED_CLAIMS` ground truth.
Loaded once by `00_main`. All architectures run the same 25 questions.

In [0]:
EVALUATION_QUESTIONS = [
    {
        "id": "Q1",
        "question": (
            "How many total certificates were earned in February 2026?"
        ),
        "expected_answer": (
            "2239 total certificates were earned in February 2026, consisting of 276 elective (Wahlzertifikat) "
            "and 1963 mandatory (Pflichtzertifikat) certificates. March was the peak month for certificate "
            "activity in 2026 with 2927 certificates."
        ),
        "difficulty": "easy",
        "question_type": "basic_aggregation",
        "tables_needed": ["mt_safe_user"],
        "key_columns": ["year", "month", "certificate_type"],
        "verification_sql": """
            SELECT COUNT(*) as total,
                   SUM(CASE WHEN certificate_type = 'Wahlzertifikat' THEN 1 ELSE 0 END) as elective,
                   SUM(CASE WHEN certificate_type = 'Pflichtzertifikat' THEN 1 ELSE 0 END) as mandatory
            FROM dev_forge_default.mt_davide.mt_safe_user
            WHERE year = 2026 AND month = 2
        """
    },

    {
        "id": "Q2",
        "question": (
            "How many distinct users earned at least one certificate per month in Q1 2026? "
            "What is the growth trend?"
        ),
        "expected_answer": (
            "Distinct users earning certificates in Q1 2026: January = 347 users, February = 434 users, "
            "March = 618 users. The average is approximately 466 users per month. There is a strong growth "
            "trend with a 78% increase from January to March, indicating growing platform engagement "
            "across the quarter."
        ),
        "difficulty": "easy",
        "question_type": "temporal_aggregation",
        "tables_needed": ["mt_safe_user"],
        "key_columns": ["year", "month", "u_id"],
        "verification_sql": """
            SELECT month, COUNT(DISTINCT u_id) as distinct_users
            FROM dev_forge_default.mt_davide.mt_safe_user
            WHERE year = 2026 AND month IN (1, 2, 3)
            GROUP BY month ORDER BY month
        """
    },

    {
        "id": "Q3",
        "question": (
            "Which 5 companies earned the most certificates in 2026? "
            "Show the company ID and total certificate count for each."
        ),
        "expected_answer": (
            "The top 5 companies by certificate count in 2026 are: "
            "1) C_fcc053f19146 with 1246 certificates, "
            "2) C_09daf3bd3e48 with 1149 certificates, "
            "3) C_b659e12972b7 with 1016 certificates, "
            "4) C_c29ab0ccbfbb with 680 certificates, "
            "5) C_534b9225f01a with 430 certificates. "
            "The distribution shows that certificate activity is concentrated "
            "among a small number of companies, with the top 3 accounting for over 75% of the top-5 total."
        ),
        "difficulty": "medium",
        "question_type": "ranking",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment"],
        "key_columns": ["company_ID1", "assignment_id1", "year"],
        "verification_sql": """
            SELECT a.company_ID1, COUNT(*) as cert_count
            FROM dev_forge_default.mt_davide.mt_safe_user u
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
            WHERE u.year = 2026
            GROUP BY a.company_ID1
            ORDER BY cert_count DESC
            LIMIT 5
        """
    },

    {
        "id": "Q4",
        "question": (
            "Compare mandatory vs elective certificate completion trends month-over-month in 2026. "
            "Which type is growing faster?"
        ),
        "expected_answer": (
            "Mandatory certificates (Pflichtzertifikat) dominate with 11904 total (93.1%) vs 881 elective "
            "(Wahlzertifikat) certificates (6.9%) through July 2026. "
            "Neither type shows consistent growth - both fluctuate month-to-month. "
            "Mandatory certificates peaked in March (2772) and elective peaked in February (276). "
            "Both types show a significant dip in July (561 mandatory, 40 elective) as the month is incomplete."
        ),
        "difficulty": "medium",
        "question_type": "trend_comparison",
        "tables_needed": ["mt_safe_user"],
        "key_columns": ["year", "month", "certificate_type"],
        "verification_sql": """
            SELECT month, certificate_type, COUNT(*) as cnt
            FROM dev_forge_default.mt_davide.mt_safe_user
            WHERE year = 2026
            GROUP BY month, certificate_type
            ORDER BY month, certificate_type
        """
    },

    {
        "id": "Q5",
        "question": (
            "What is the on-time completion rate for mandatory courses (Pflichtkurs) by company type in 2026?"
        ),
        "expected_answer": (
            "On-time completion rates for mandatory courses (Pflichtkurs) by company type in 2026: "
            "Baercare companies have the highest rate at 75.8% (2154 on-time out of 2840 total), "
            "followed by Care companies at 55.5% (233 on-time out of 420 total). "
            "Baercare organizations show significantly better deadline compliance than Care organizations."
        ),
        "difficulty": "medium",
        "question_type": "segmented_metric",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment", "mt_safe_company", "mt_safe_company_type"],
        "key_columns": ["type_name", "course_type", "completedOn_ts", "deadline_assignment"],
        "verification_sql": """
            SELECT ct.type_name, COUNT(*) as total,
                   SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) as on_time,
                   ROUND(SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as on_time_rate
            FROM dev_forge_default.mt_davide.mt_safe_user u
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
            JOIN dev_forge_default.mt_davide.mt_safe_company c ON a.company_ID1 = c.company_ID
            JOIN dev_forge_default.mt_davide.mt_safe_company_type ct ON c.subTypeId = ct.type_id
            WHERE u.course_type = 'Pflichtkurs' AND u.year = 2026
              AND a.deadline_assignment IS NOT NULL AND u.completedOn_ts IS NOT NULL
            GROUP BY ct.type_name ORDER BY on_time_rate DESC
        """
    },

    {
        "id": "Q6",
        "question": (
            "Which 5 active companies have the highest certificate volumes in 2026? "
            "Show their company ID, certificate count, and on-time completion rate."
        ),
        "expected_answer": (
            "The top 5 active companies by certificate volume in 2026 are: "
            "1) C_fcc053f19146 with 1246 certificates (92.5% on-time), "
            "2) C_09daf3bd3e48 with 1149 certificates (37.6% on-time), "
            "3) C_b659e12972b7 with 1016 certificates (100.0% on-time), "
            "4) C_c29ab0ccbfbb with 680 certificates (61.6% on-time), "
            "5) C_534b9225f01a with 430 certificates (0.2% on-time). "
            "There is no clear inverse correlation between volume and compliance - "
            "high-volume companies show widely varying on-time rates from 0.2% to 100%."
        ),
        "difficulty": "hard",
        "question_type": "multi_metric_correlation",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment", "mt_safe_company"],
        "key_columns": ["company_ID1", "company_status_simple", "completedOn_ts", "deadline_assignment"],
        "verification_sql": """
            SELECT a.company_ID1, COUNT(*) as cert_volume,
                   ROUND(SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as on_time_rate
            FROM dev_forge_default.mt_davide.mt_safe_user u
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
            JOIN dev_forge_default.mt_davide.mt_safe_company c ON a.company_ID1 = c.company_ID
            WHERE u.year = 2026 AND a.deadline_assignment IS NOT NULL
              AND u.completedOn_ts IS NOT NULL AND c.company_status_simple = 'ACTIVE'
            GROUP BY a.company_ID1 HAVING COUNT(*) >= 20
            ORDER BY cert_volume DESC LIMIT 5
        """
    },

    {
        "id": "Q7",
        "question": (
            "How has the on-time completion rate changed month-over-month in H1 2026 "
            "(January through June)? Identify the best and worst performing months."
        ),
        "expected_answer": (
            "On-time completion rates in H1 2026: "
            "January 55.7%, February 56.8%, March 71.0% (best), April 67.1%, May 70.9%, June 68.4%. "
            "January is the worst month (55.7%) while March has the highest compliance (71.0%). "
            "The rate improves significantly from Jan/Feb to Mar and remains elevated through June. "
            "The overall trend shows improvement after a slower start to the year."
        ),
        "difficulty": "hard",
        "question_type": "derived_metric_trend",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment"],
        "key_columns": ["month", "completedOn_ts", "deadline_assignment"],
        "verification_sql": """
            SELECT u.month, COUNT(*) as total,
                   SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) as on_time,
                   ROUND(SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as on_time_rate
            FROM dev_forge_default.mt_davide.mt_safe_user u
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
            WHERE u.year = 2026 AND u.month <= 6
              AND a.deadline_assignment IS NOT NULL AND u.completedOn_ts IS NOT NULL
            GROUP BY u.month ORDER BY u.month
        """
    },

    {
        "id": "Q8",
        "question": (
            "For companies with INACTIVE status, what was their average on-time completion rate "
            "compared to ACTIVE companies? Use compliance data from 2026."
        ),
        "expected_answer": (
            "INACTIVE companies had a higher on-time completion rate (90.7%) compared to ACTIVE companies (65.0%) in 2026. "
            "However, this is misleading because INACTIVE companies generated only 194 completions versus 12013 for ACTIVE. "
            "The high rate for INACTIVE likely reflects survivorship bias - only the most compliant assignments "
            "were completed before inactivation. ACTIVE companies have 65% on-time across a much larger volume."
        ),
        "difficulty": "hard",
        "question_type": "churn_analysis",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment", "mt_safe_company"],
        "key_columns": ["company_status_simple", "completedOn_ts", "deadline_assignment"],
        "verification_sql": """
            SELECT c.company_status_simple, COUNT(*) as total_certs,
                   SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) as on_time,
                   ROUND(SUM(CASE WHEN u.completedOn_ts <= a.deadline_assignment THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as on_time_rate
            FROM dev_forge_default.mt_davide.mt_safe_user u
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
            JOIN dev_forge_default.mt_davide.mt_safe_company c ON a.company_ID1 = c.company_ID
            WHERE u.year = 2026 AND a.deadline_assignment IS NOT NULL AND u.completedOn_ts IS NOT NULL
            GROUP BY c.company_status_simple ORDER BY total_certs DESC
        """
    },

    {
        "id": "Q9",
        "question": (
            "Identify companies (by ID) where certificate completion dropped by more than 50% "
            "month-over-month at least twice in 2026. How many such companies exist and what "
            "is their total certificate volume?"
        ),
        "expected_answer": (
            "30 companies experienced 50%+ month-over-month certificate drops at least twice in 2026. "
            "Their combined certificate volume is 6857. The top 3 by volume are: "
            "C_fcc053f19146 (1246 certs, 2 drops), C_09daf3bd3e48 (1149 certs, 2 drops), "
            "C_b659e12972b7 (1016 certs, 3 drops). "
            "Many drops occur in month 7 (July) due to incomplete data for the partial month. "
            "This analysis requires LAG() window functions and CTE-based filtering."
        ),
        "difficulty": "very_hard",
        "question_type": "temporal_pattern_detection",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment"],
        "key_columns": ["company_ID1", "month", "assignment_id1"],
        "verification_sql": """
            WITH monthly AS (
                SELECT a.company_ID1, u.month, COUNT(*) as cnt
                FROM dev_forge_default.mt_davide.mt_safe_user u
                JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON u.assignment_id1 = a.assignment_ID
                WHERE u.year = 2026
                GROUP BY a.company_ID1, u.month
            ),
            with_prev AS (
                SELECT *, LAG(cnt) OVER (PARTITION BY company_ID1 ORDER BY month) as prev_cnt
                FROM monthly
            ),
            drops AS (
                SELECT company_ID1 FROM with_prev
                WHERE prev_cnt > 0 AND (prev_cnt - cnt) * 100.0 / prev_cnt >= 50
            ),
            companies_twice AS (
                SELECT company_ID1, COUNT(*) as num_drops
                FROM drops GROUP BY company_ID1 HAVING COUNT(*) >= 2
            )
            SELECT ct.company_ID1, ct.num_drops, SUM(m.cnt) as total_certs
            FROM companies_twice ct
            JOIN monthly m ON ct.company_ID1 = m.company_ID1
            GROUP BY ct.company_ID1, ct.num_drops
            ORDER BY total_certs DESC
        """
    },

    {
        "id": "Q10",
        "question": (
            "Build a monthly cohort analysis: for companies created in each month of 2026, "
            "track their total certificate output. Which cohort (by creation month) "
            "showed the strongest early engagement?"
        ),
        "expected_answer": (
            "Monthly cohort analysis for companies created in 2026: "
            "January cohort: 5 companies, 399 certificates (79.8 avg per company). "
            "February cohort: 8 companies, 267 certificates (33.4 avg). "
            "March cohort: 8 companies, 687 certificates (85.9 avg - strongest engagement). "
            "April cohort: 3 companies, 43 certificates (14.3 avg). "
            "May cohort: 4 companies, 112 certificates (28.0 avg). "
            "June cohort: 8 companies, 41 certificates (5.1 avg). "
            "The March cohort showed the strongest early engagement with 85.9 avg certificates per company."
        ),
        "difficulty": "very_hard",
        "question_type": "cohort_analysis",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment", "mt_safe_company"],
        "key_columns": ["company_ID", "company_created_at", "assignment_id1"],
        "verification_sql": """
            WITH cohorts AS (
                SELECT company_ID, MONTH(company_created_at) as cohort_month
                FROM dev_forge_default.mt_davide.mt_safe_company
                WHERE YEAR(company_created_at) = 2026
            )
            SELECT co.cohort_month,
                   COUNT(DISTINCT co.company_ID) as companies_in_cohort,
                   COUNT(*) as total_certs,
                   ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT co.company_ID), 1) as avg_per_company
            FROM cohorts co
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON co.company_ID = a.company_ID1
            JOIN dev_forge_default.mt_davide.mt_safe_user u ON u.assignment_id1 = a.assignment_ID
            WHERE u.year = 2026
            GROUP BY co.cohort_month ORDER BY co.cohort_month
        """
    }
]

print(f"Loaded {len(EVALUATION_QUESTIONS)} evaluation questions")
print(f"\nDifficulty distribution:")
for d in ['easy', 'medium', 'hard', 'very_hard']:
    count = sum(1 for q in EVALUATION_QUESTIONS if q['difficulty'] == d)
    if count > 0:
        print(f"  {d}: {count}")
print(f"\nQuestions:")
for q in EVALUATION_QUESTIONS:
    print(f"  {q['id']} [{q['difficulty']}] {q['question_type']}: {q['question'][:70]}...")

Loaded 10 evaluation questions

Difficulty distribution:
  easy: 2
  medium: 3
  hard: 3
  very_hard: 2

Questions:
  Q1 [easy] basic_aggregation: How many total certificates were earned in February 2026?...
  Q2 [easy] temporal_aggregation: What was the average monthly revenue in Q1 2026?...
  Q3 [medium] ranking: Which 5 companies earned the most certificates in 2026? Show the compa...
  Q4 [medium] trend_comparison: Compare mandatory vs elective certificate completion trends month-over...
  Q5 [medium] segmented_metric: What is the on-time completion rate for mandatory courses (Pflichtkurs...
  Q6 [hard] multi_metric_correlation: Which 5 active companies have the lowest on-time completion rates but ...
  Q7 [hard] derived_metric_trend: Calculate the revenue per certificate earned for each month in 2026. H...
  Q8 [hard] churn_analysis: For companies with INACTIVE status, what was their average on-time com...
  Q9 [very_hard] temporal_pattern_detection: Identify companies (by ID) whe

In [0]:
# =============================================================================
# EXTENDED EVALUATION QUESTIONS (Q11-Q25)
# Design principles:
#   - 5 UNANSWERABLE questions (hallucination tests) — data doesn't exist
#   - Questions targeting architecture-specific weaknesses:
#       SAS: context overload, multi-table joins without retrieval hints
#       RAG: questions where top-k retrieval may miss relevant chunks
#       MAS: simple questions where coordination overhead hurts
#   - No redundancy with Q1-Q10
# =============================================================================

EVALUATION_QUESTIONS.extend([
    # =========================================================================
    # EASY — Q11, Q12, Q13
    # =========================================================================
    {
        "id": "Q11",
        "question": (
            "How many companies are currently in ACTIVE status? "
            "How does that compare to the total number of companies?"
        ),
        "expected_answer": (
            "The number of ACTIVE companies can be found by filtering mt_safe_company by company_Status = 'ACTIVE'. "
            "The result shows the ratio of active to total companies, indicating the platform's customer retention."
        ),
        "difficulty": "easy",
        "question_type": "basic_count",
        "tables_needed": ["mt_safe_company"],
        "key_columns": ["company_ID", "company_Status"],
        "answerability_label": "answerable",
        "architecture_target": "MAS_weakness",  # Simple query — MAS coordination overhead hurts
        "verification_sql": """
            SELECT company_Status, COUNT(*) as company_count
            FROM dev_forge_default.mt_davide.mt_safe_company
            GROUP BY company_Status
            ORDER BY company_count DESC
        """
    },

    {
        "id": "Q12",
        "question": (
            "What was the average monthly revenue per user in Q1 2026? "
            "Break it down by company size category."
        ),
        "expected_answer": (
            "This question CANNOT be answered with the available data. "
            "The mt_safe_* tables do not contain any revenue, financial, or billing information. "
            "Only training/learning management data (certificates, assignments, courses, exams) is available. "
            "A correct response should explicitly state that revenue data is not present in the schema."
        ),
        "difficulty": "easy",
        "question_type": "unanswerable_financial",
        "tables_needed": [],
        "key_columns": [],
        "answerability_label": "unanswerable",
        "architecture_target": "hallucination_test",
        "verification_sql": "-- No revenue data exists in mt_safe_* tables"
    },

    {
        "id": "Q13",
        "question": (
            "How many distinct courses exist in the platform? "
            "What is the split between mandatory (Pflichtkurs) and elective (Wahlkurs) courses?"
        ),
        "expected_answer": (
            "The total number of distinct courses and their type split (Pflichtkurs vs Wahlkurs) "
            "can be found from mt_safe_course. The result should show the count of each course_type "
            "and the percentage breakdown."
        ),
        "difficulty": "easy",
        "question_type": "basic_aggregation",
        "tables_needed": ["mt_safe_user"],
        "key_columns": ["course_id", "course_type"],
        "answerability_label": "answerable",
        "architecture_target": "MAS_weakness",  # Very simple — single table, basic GROUP BY
        "verification_sql": """
            SELECT course_type, COUNT(DISTINCT course_id) as course_count
            FROM dev_forge_default.mt_davide.mt_safe_course
            GROUP BY course_type
        """
    },

    # =========================================================================
    # MEDIUM — Q14, Q15, Q16, Q17, Q18
    # =========================================================================
    {
        "id": "Q14",
        "question": (
            "Which companies created their account most recently (in 2026)? "
            "Show the top 5 newest companies with their creation date and current status."
        ),
        "expected_answer": (
            "The newest companies can be found by sorting mt_safe_company by company_created_at DESC. "
            "The result shows the 5 most recently created company accounts, their IDs, creation dates, "
            "and current status (ACTIVE/INACTIVE/TRIAL)."
        ),
        "difficulty": "medium",
        "question_type": "sorting_filtering",
        "tables_needed": ["mt_safe_company"],
        "key_columns": ["company_ID", "company_created_at", "company_Status"],
        "answerability_label": "answerable",
        "architecture_target": "RAG_weakness",  # Requires knowing company_created_at exists — may not be in top-k chunks
        "verification_sql": """
            SELECT company_ID, company_Name, company_created_at, company_Status
            FROM dev_forge_default.mt_davide.mt_safe_company
            WHERE YEAR(company_created_at) = 2026
            ORDER BY company_created_at DESC
            LIMIT 5
        """
    },

    {
        "id": "Q15",
        "question": (
            "What is the geographic distribution of companies by country and region? "
            "Which region has the highest certificate completion rate?"
        ),
        "expected_answer": (
            "This question CANNOT be answered with the available data. "
            "The mt_safe_* tables do not contain any geographic information (country, region, city, address). "
            "Company data includes only ID, name, status, and creation date — no location fields. "
            "A correct response should explicitly state that geographic data is not available."
        ),
        "difficulty": "medium",
        "question_type": "unanswerable_geographic",
        "tables_needed": [],
        "key_columns": [],
        "answerability_label": "unanswerable",
        "architecture_target": "hallucination_test",
        "verification_sql": "-- No geographic data exists in mt_safe_* tables"
    },

    {
        "id": "Q16",
        "question": (
            "What percentage of assignments with a deadline were completed late in 2026? "
            "How many assignments had no deadline at all?"
        ),
        "expected_answer": (
            "This requires comparing assignment_completed_at vs deadline_assignment in mt_safe_assignment. "
            "Late completions are those where assignment_completed_at > deadline_assignment. "
            "Assignments with NULL deadline_assignment have no deadline. "
            "The result should show: total assignments, assignments with deadline, late count, late %, and no-deadline count."
        ),
        "difficulty": "medium",
        "question_type": "conditional_aggregation",
        "tables_needed": ["mt_safe_assignment"],
        "key_columns": ["assignment_ID", "assignment_completed_at", "deadline_assignment"],
        "answerability_label": "answerable",
        "architecture_target": "SAS_weakness",  # Needs precise column names; SAS may confuse with full context dump
        "verification_sql": """
            SELECT
                COUNT(*) as total_assignments,
                SUM(CASE WHEN deadline_assignment IS NOT NULL THEN 1 ELSE 0 END) as with_deadline,
                SUM(CASE WHEN deadline_assignment IS NULL THEN 1 ELSE 0 END) as no_deadline,
                SUM(CASE WHEN assignment_completed_at > deadline_assignment THEN 1 ELSE 0 END) as late_count,
                ROUND(SUM(CASE WHEN assignment_completed_at > deadline_assignment THEN 1 ELSE 0 END) * 100.0 /
                      NULLIF(SUM(CASE WHEN deadline_assignment IS NOT NULL THEN 1 ELSE 0 END), 0), 1) as late_pct
            FROM dev_forge_default.mt_davide.mt_safe_assignment
            WHERE YEAR(assignment_completed_at) = 2026 OR YEAR(deadline_assignment) = 2026
        """
    },

    {
        "id": "Q17",
        "question": (
            "For each company size category, what is the average number of distinct courses "
            "assigned per company? Which size category assigns the most diverse course portfolio?"
        ),
        "expected_answer": (
            "This requires joining mt_safe_company_type (company_size_category) with mt_safe_assignment "
            "or mt_safe_assignment_course_progress to count distinct courses per company, then averaging "
            "by size category. Larger companies likely assign more diverse courses."
        ),
        "difficulty": "medium",
        "question_type": "multi_table_join",
        "tables_needed": ["mt_safe_company_type", "mt_safe_company", "mt_safe_assignment", "mt_safe_assignment_course_progress"],
        "key_columns": ["type_name", "subTypeId", "company_ID1", "course_id"],
        "answerability_label": "answerable",
        "architecture_target": "SAS_weakness",  # Multi-table join with ambiguous join keys — SAS may use wrong columns
        "verification_sql": """
            SELECT ct.company_size_category,
                   COUNT(DISTINCT acp.company_ID1) as num_companies,
                   COUNT(DISTINCT acp.course_id) as total_distinct_courses,
                   ROUND(COUNT(DISTINCT acp.course_id) * 1.0 / COUNT(DISTINCT acp.company_ID1), 1) as avg_courses_per_company
            FROM dev_forge_default.mt_davide.mt_safe_assignment_course_progress acp
            JOIN dev_forge_default.mt_davide.mt_safe_company_type ct
              ON acp.company_ID1 = ct.company_ID
            GROUP BY ct.company_size_category
            ORDER BY avg_courses_per_company DESC
        """
    },

    {
        "id": "Q18",
        "question": (
            "What is the average training budget per employee for each company size category? "
            "Which companies exceeded their allocated training budget in 2026?"
        ),
        "expected_answer": (
            "This question CANNOT be answered with the available data. "
            "The mt_safe_* tables do not contain any financial, budget, or cost information. "
            "Training budgets, employee headcounts, and spending data are not tracked in this LMS dataset. "
            "Only learning activity data (assignments, completions, certificates, exams) is available."
        ),
        "difficulty": "medium",
        "question_type": "unanswerable_budget",
        "tables_needed": [],
        "key_columns": [],
        "answerability_label": "unanswerable",
        "architecture_target": "hallucination_test",
        "verification_sql": "-- No budget/financial data exists in mt_safe_* tables"
    },

    # =========================================================================
    # HARD — Q19, Q20, Q21, Q22
    # =========================================================================
    {
        "id": "Q19",
        "question": (
            "What is the certificate-to-assignment ratio for each company in 2026? "
            "Which 5 companies have the highest conversion rate (certificates earned / assignments given)?"
        ),
        "expected_answer": (
            "This requires joining mt_safe_certificate (or mt_safe_user for certificate records) with "
            "mt_safe_assignment to compute certificates/assignments per company. "
            "A high ratio indicates efficient completion; a low ratio suggests assignments go uncompleted. "
            "The top 5 by conversion rate should be identified by company_ID."
        ),
        "difficulty": "hard",
        "question_type": "derived_ratio_ranking",
        "tables_needed": ["mt_safe_user", "mt_safe_assignment"],
        "key_columns": ["company_ID1", "certificate_type", "assignment_ID"],
        "answerability_label": "answerable",
        "architecture_target": "RAG_weakness",  # Requires combining info from multiple semantic chunks
        "verification_sql": """
            WITH certs AS (
                SELECT company_ID1, COUNT(*) as cert_count
                FROM dev_forge_default.mt_davide.mt_safe_user
                WHERE year = 2026 AND certificate_type IS NOT NULL
                GROUP BY company_ID1
            ),
            assigns AS (
                SELECT company_ID1, COUNT(*) as assign_count
                FROM dev_forge_default.mt_davide.mt_safe_assignment
                WHERE YEAR(assignment_completed_at) = 2026 OR YEAR(deadline_assignment) = 2026
                GROUP BY company_ID1
            )
            SELECT c.company_ID1,
                   c.cert_count,
                   a.assign_count,
                   ROUND(c.cert_count * 1.0 / NULLIF(a.assign_count, 0), 3) as conversion_rate
            FROM certs c
            JOIN assigns a ON c.company_ID1 = a.company_ID1
            WHERE a.assign_count > 0
            ORDER BY conversion_rate DESC
            LIMIT 5
        """
    },

    {
        "id": "Q20",
        "question": (
            "What is the average time-to-completion (days between assignment creation and completion) "
            "for mandatory vs elective courses? Do larger companies complete faster?"
        ),
        "expected_answer": (
            "This requires computing DATEDIFF between assignment_completed_at and assignment creation (or deadline_assignment) "
            "in mt_safe_assignment, grouped by course_type and company_size_category (from mt_safe_company_type). "
            "Mandatory courses (Pflichtkurs) likely have shorter time-to-completion due to deadlines. "
            "Larger companies may complete faster due to structured L&D programs."
        ),
        "difficulty": "hard",
        "question_type": "time_analysis",
        "tables_needed": ["mt_safe_assignment", "mt_safe_company", "mt_safe_company_type"],
        "key_columns": ["assignment_completed_at", "deadline_assignment", "status_assignment", "type_name"],
        "answerability_label": "answerable",
        "architecture_target": "SAS_weakness",  # Complex date math + multi-table join — context overload for SAS
        "verification_sql": """
            SELECT ct.company_size_category,
                   a.status_assignment,
                   COUNT(*) as n_assignments,
                   ROUND(AVG(DATEDIFF(a.assignment_completed_at, a.deadline_assignment)), 1) as avg_days_vs_deadline
            FROM dev_forge_default.mt_davide.mt_safe_assignment a
            JOIN dev_forge_default.mt_davide.mt_safe_company_type ct ON a.company_ID1 = ct.company_ID
            WHERE a.assignment_completed_at IS NOT NULL
            GROUP BY ct.company_size_category, a.status_assignment
            ORDER BY ct.company_size_category, a.status_assignment
        """
    },

    {
        "id": "Q21",
        "question": (
            "What is the average learner satisfaction score (NPS) by course type and company size? "
            "Which courses have the lowest satisfaction ratings in 2026?"
        ),
        "expected_answer": (
            "This question CANNOT be answered with the available data. "
            "The mt_safe_* tables do not contain any satisfaction scores, NPS ratings, feedback, "
            "or survey data. The LMS data only tracks completions, certificates, exams, and assignments — "
            "not qualitative learner sentiment. A correct response should state this limitation clearly."
        ),
        "difficulty": "hard",
        "question_type": "unanswerable_satisfaction",
        "tables_needed": [],
        "key_columns": [],
        "answerability_label": "unanswerable",
        "architecture_target": "hallucination_test",
        "verification_sql": "-- No satisfaction/NPS/feedback data exists in mt_safe_* tables"
    },

    {
        "id": "Q22",
        "question": (
            "Which companies have active assignments but earned zero certificates in 2026? "
            "How many total assignments do these 'non-converting' companies have?"
        ),
        "expected_answer": (
            "This requires finding companies present in mt_safe_assignment (with 2026 activity) "
            "that are NOT present in mt_safe_user (certificate records) for 2026. "
            "These are companies where users are assigned courses but never complete them to earn certificates. "
            "The result should list company IDs and their assignment counts. "
            "This pattern indicates potential engagement issues or recently onboarded companies."
        ),
        "difficulty": "hard",
        "question_type": "negation_set_difference",
        "tables_needed": ["mt_safe_assignment", "mt_safe_user"],
        "key_columns": ["company_ID1"],
        "answerability_label": "answerable",
        "architecture_target": "RAG_weakness",  # Negation/NOT IN logic — RAG chunks don't teach anti-join patterns
        "verification_sql": """
            WITH active_assign_companies AS (
                SELECT DISTINCT company_ID1
                FROM dev_forge_default.mt_davide.mt_safe_assignment
                WHERE YEAR(assignment_completed_at) = 2026 OR YEAR(deadline_assignment) = 2026
            ),
            cert_companies AS (
                SELECT DISTINCT company_ID1
                FROM dev_forge_default.mt_davide.mt_safe_user
                WHERE year = 2026 AND certificate_type IS NOT NULL
            )
            SELECT ac.company_ID1,
                   COUNT(a.assignment_ID) as total_assignments
            FROM active_assign_companies ac
            LEFT JOIN cert_companies cc ON ac.company_ID1 = cc.company_ID1
            JOIN dev_forge_default.mt_davide.mt_safe_assignment a ON ac.company_ID1 = a.company_ID1
            WHERE cc.company_ID1 IS NULL
            GROUP BY ac.company_ID1
            ORDER BY total_assignments DESC
        """
    },

    # =========================================================================
    # VERY_HARD — Q23, Q24, Q25
    # =========================================================================
    {
        "id": "Q23",
        "question": (
            "Identify users who earned certificates in at least 3 consecutive months in 2026. "
            "How many such 'consistently active' users exist, and what percentage of total certificate-earning users do they represent?"
        ),
        "expected_answer": (
            "This requires a window-function approach: for each user, identify monthly certificate activity, "
            "then detect sequences of 3+ consecutive months using ROW_NUMBER() tricks or LEAD/LAG. "
            "The result should show the count of consistently active users vs total users, "
            "plus their share of overall certificate volume. These power users likely drive disproportionate platform value."
        ),
        "difficulty": "very_hard",
        "question_type": "consecutive_pattern_detection",
        "tables_needed": ["mt_safe_user"],
        "key_columns": ["u_id", "year", "month", "certificate_type"],
        "answerability_label": "answerable",
        "architecture_target": "all_struggle",  # Complex window functions — all architectures face SQL generation challenge
        "verification_sql": """
            WITH user_months AS (
                SELECT u_id, year, month,
                       ROW_NUMBER() OVER (PARTITION BY u_id ORDER BY year, month) as rn
                FROM dev_forge_default.mt_davide.mt_safe_user
                WHERE year = 2026 AND certificate_type IS NOT NULL
                GROUP BY u_id, year, month
            ),
            streaks AS (
                SELECT u_id, month - rn as streak_group,
                       COUNT(*) as consecutive_months
                FROM user_months
                GROUP BY u_id, month - rn
            ),
            consistent_users AS (
                SELECT DISTINCT u_id
                FROM streaks
                WHERE consecutive_months >= 3
            )
            SELECT
                (SELECT COUNT(*) FROM consistent_users) as consistent_user_count,
                (SELECT COUNT(DISTINCT u_id) FROM dev_forge_default.mt_davide.mt_safe_user WHERE year = 2026 AND certificate_type IS NOT NULL) as total_cert_users,
                ROUND((SELECT COUNT(*) FROM consistent_users) * 100.0 /
                      NULLIF((SELECT COUNT(DISTINCT u_id) FROM dev_forge_default.mt_davide.mt_safe_user WHERE year = 2026 AND certificate_type IS NOT NULL), 0), 1) as pct
        """
    },

    {
        "id": "Q24",
        "question": (
            "Which instructors or trainers have the highest course completion rates? "
            "Show the top 10 instructors by average learner success rate and total courses taught."
        ),
        "expected_answer": (
            "This question CANNOT be answered with the available data. "
            "The mt_safe_* tables do not contain any instructor, trainer, or teacher information. "
            "Courses are tracked only by ID and type — there is no instructor_id, trainer_name, or "
            "facilitator field in any table. The LMS data captures learner activity only, not who delivers the training. "
            "A correct response must state that instructor data is not available."
        ),
        "difficulty": "very_hard",
        "question_type": "unanswerable_instructor",
        "tables_needed": [],
        "key_columns": [],
        "answerability_label": "unanswerable",  # 5th unanswerable — catches overconfident complex SQL generation
        "architecture_target": "hallucination_test",
        "verification_sql": "-- No instructor/trainer data exists in mt_safe_* tables"
    },

    {
        "id": "Q25",
        "question": (
            "Detect anomalous companies: find companies whose exam failure rate in 2026 is more than "
            "2 standard deviations above the overall average failure rate. "
            "Show company ID, their failure rate, the overall average, and the threshold."
        ),
        "expected_answer": (
            "This requires computing per-company exam failure rates from mt_safe_exam_result, "
            "calculating the global mean and standard deviation, then filtering companies whose rate "
            "exceeds mean + 2*stddev. The query uses CTEs: first compute per-company fail rates, "
            "then compute global stats, then filter anomalies. Companies with significantly higher "
            "failure rates may need additional support or have issues with course difficulty matching."
        ),
        "difficulty": "very_hard",
        "question_type": "statistical_anomaly_detection",
        "tables_needed": ["mt_safe_exam_result", "mt_safe_assignment"],
        "key_columns": ["exam_assignmentId", "hasPassed", "resultPercentage", "company_ID1"],
        "answerability_label": "answerable",
        "architecture_target": "all_struggle",  # Statistical query with stddev — very hard for all architectures
        "verification_sql": """
            WITH company_failure AS (
                SELECT company_ID1,
                       COUNT(*) as total_exams,
                       SUM(CASE WHEN exam_passed = false OR exam_result < 60 THEN 1 ELSE 0 END) as failed_exams,
                       ROUND(SUM(CASE WHEN exam_passed = false OR exam_result < 60 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as failure_rate
                FROM dev_forge_default.mt_davide.mt_safe_exam_result
                WHERE YEAR(exam_date) = 2026
                GROUP BY company_ID1
                HAVING COUNT(*) >= 5  -- minimum sample size
            ),
            stats AS (
                SELECT AVG(failure_rate) as mean_rate,
                       STDDEV(failure_rate) as std_rate
                FROM company_failure
            )
            SELECT cf.company_ID1, cf.total_exams, cf.failure_rate,
                   s.mean_rate, s.mean_rate + 2 * s.std_rate as threshold
            FROM company_failure cf
            CROSS JOIN stats s
            WHERE cf.failure_rate > s.mean_rate + 2 * s.std_rate
            ORDER BY cf.failure_rate DESC
        """
    },
])

# --- Summary ---
print(f"\n{'='*70}")
print(f"EXTENDED EVALUATION SET: {len(EVALUATION_QUESTIONS)} total questions")
print(f"{'='*70}")
print(f"\nDifficulty distribution:")
for d in ['easy', 'medium', 'hard', 'very_hard']:
    count = sum(1 for q in EVALUATION_QUESTIONS if q['difficulty'] == d)
    print(f"  {d}: {count}")

unanswerable = sum(1 for q in EVALUATION_QUESTIONS if q.get('answerability_label') == 'unanswerable')
print(f"\nUnanswerable (hallucination tests): {unanswerable}")
print(f"Answerable: {len(EVALUATION_QUESTIONS) - unanswerable}")
print(f"\nNew questions:")
for q in EVALUATION_QUESTIONS[10:]:
    label = ' [UNANSWERABLE]' if q.get('answerability_label') == 'unanswerable' else ''
    target = f" → {q.get('architecture_target', '')}" if q.get('architecture_target') else ''
    print(f"  {q['id']} [{q['difficulty']}] {q['question_type']}{label}{target}: {q['question'][:60]}...")

In [0]:
# =============================================================================
# EXPECTED CLAIMS — Ground-truth atomic claims per evaluation question
# Manually reviewed and persisted. Used by claim-level P/R/F1 evaluation.
# Do NOT regenerate during architecture runs.
# =============================================================================

_EXPECTED_CLAIMS = {
    "Q1": [
        {"claim_id": "Q1_E1", "text": "2239 total certificates were earned in February 2026", "entity": None, "metric": "certificate count", "value": 2239, "unit": None, "period": "february 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q1_E2", "text": "276 elective (Wahlzertifikat) certificates were earned in February 2026", "entity": None, "metric": "certificate count", "value": 276, "unit": None, "period": "february 2026", "qualifiers": {"certificate_type": "Wahlzertifikat"}, "required": True},
        {"claim_id": "Q1_E3", "text": "1963 mandatory (Pflichtzertifikat) certificates were earned in February 2026", "entity": None, "metric": "certificate count", "value": 1963, "unit": None, "period": "february 2026", "qualifiers": {"certificate_type": "Pflichtzertifikat"}, "required": True},
    ],
    "Q2": [
        {"claim_id": "Q2_E1", "text": "347 distinct users earned certificates in January 2026", "entity": None, "metric": "distinct users", "value": 347, "unit": None, "period": "january 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q2_E2", "text": "434 distinct users earned certificates in February 2026", "entity": None, "metric": "distinct users", "value": 434, "unit": None, "period": "february 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q2_E3", "text": "618 distinct users earned certificates in March 2026", "entity": None, "metric": "distinct users", "value": 618, "unit": None, "period": "march 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q2_E4", "text": "The average is approximately 466 users per month in Q1 2026", "entity": None, "metric": "distinct users", "value": 466, "unit": None, "period": "q1 2026", "qualifiers": {"aggregation": "average"}, "required": True},
        {"claim_id": "Q2_E5", "text": "There is a 78% increase from January to March 2026", "entity": None, "metric": "growth rate", "value": 78, "unit": "%", "period": "january to march 2026", "qualifiers": {}, "required": True},
    ],
    "Q3": [
        {"claim_id": "Q3_E1", "text": "C_fcc053f19146 earned 1246 certificates in 2026", "entity": "c_fcc053f19146", "metric": "certificate count", "value": 1246, "unit": None, "period": "2026", "qualifiers": {"rank": 1}, "required": True},
        {"claim_id": "Q3_E2", "text": "C_09daf3bd3e48 earned 1149 certificates in 2026", "entity": "c_09daf3bd3e48", "metric": "certificate count", "value": 1149, "unit": None, "period": "2026", "qualifiers": {"rank": 2}, "required": True},
        {"claim_id": "Q3_E3", "text": "C_b659e12972b7 earned 1016 certificates in 2026", "entity": "c_b659e12972b7", "metric": "certificate count", "value": 1016, "unit": None, "period": "2026", "qualifiers": {"rank": 3}, "required": True},
        {"claim_id": "Q3_E4", "text": "C_c29ab0ccbfbb earned 680 certificates in 2026", "entity": "c_c29ab0ccbfbb", "metric": "certificate count", "value": 680, "unit": None, "period": "2026", "qualifiers": {"rank": 4}, "required": True},
        {"claim_id": "Q3_E5", "text": "C_534b9225f01a earned 430 certificates in 2026", "entity": "c_534b9225f01a", "metric": "certificate count", "value": 430, "unit": None, "period": "2026", "qualifiers": {"rank": 5}, "required": True},
    ],
    "Q4": [
        {"claim_id": "Q4_E1", "text": "Mandatory certificates (Pflichtzertifikat) total 11904 through July 2026", "entity": None, "metric": "certificate count", "value": 11904, "unit": None, "period": "through july 2026", "qualifiers": {"certificate_type": "Pflichtzertifikat"}, "required": True},
        {"claim_id": "Q4_E2", "text": "Mandatory certificates account for 93.1% of total", "entity": None, "metric": "share", "value": 93.1, "unit": "%", "period": "through july 2026", "qualifiers": {"certificate_type": "Pflichtzertifikat"}, "required": True},
        {"claim_id": "Q4_E3", "text": "881 elective (Wahlzertifikat) certificates earned through July 2026", "entity": None, "metric": "certificate count", "value": 881, "unit": None, "period": "through july 2026", "qualifiers": {"certificate_type": "Wahlzertifikat"}, "required": True},
        {"claim_id": "Q4_E4", "text": "Elective certificates account for 6.9% of total", "entity": None, "metric": "share", "value": 6.9, "unit": "%", "period": "through july 2026", "qualifiers": {"certificate_type": "Wahlzertifikat"}, "required": True},
        {"claim_id": "Q4_E5", "text": "Mandatory certificates peaked in March with 2772", "entity": None, "metric": "certificate count", "value": 2772, "unit": None, "period": "march 2026", "qualifiers": {"certificate_type": "Pflichtzertifikat", "superlative": "peak"}, "required": True},
        {"claim_id": "Q4_E6", "text": "Elective certificates peaked in February with 276", "entity": None, "metric": "certificate count", "value": 276, "unit": None, "period": "february 2026", "qualifiers": {"certificate_type": "Wahlzertifikat", "superlative": "peak"}, "required": True},
    ],
    "Q5": [
        {"claim_id": "Q5_E1", "text": "Baercare companies have the highest on-time rate at 75.8%", "entity": "baercare", "metric": "on-time completion rate", "value": 75.8, "unit": "%", "period": "2026", "qualifiers": {"course_type": "Pflichtkurs"}, "required": True},
        {"claim_id": "Q5_E2", "text": "Care companies have on-time rate at 55.5%", "entity": "care", "metric": "on-time completion rate", "value": 55.5, "unit": "%", "period": "2026", "qualifiers": {"course_type": "Pflichtkurs"}, "required": True},
    ],
    "Q6": [
        {"claim_id": "Q6_E1", "text": "C_fcc053f19146 has 1246 certificates with 92.5% on-time rate", "entity": "c_fcc053f19146", "metric": "certificate count", "value": 1246, "unit": None, "period": "2026", "qualifiers": {"rank": 1}, "required": True},
        {"claim_id": "Q6_E2", "text": "C_09daf3bd3e48 has 1149 certificates with 37.6% on-time rate", "entity": "c_09daf3bd3e48", "metric": "certificate count", "value": 1149, "unit": None, "period": "2026", "qualifiers": {"rank": 2}, "required": True},
        {"claim_id": "Q6_E3", "text": "C_b659e12972b7 has 1016 certificates with 100% on-time rate", "entity": "c_b659e12972b7", "metric": "certificate count", "value": 1016, "unit": None, "period": "2026", "qualifiers": {"rank": 3}, "required": True},
        {"claim_id": "Q6_E4", "text": "C_c29ab0ccbfbb has 680 certificates with 61.6% on-time rate", "entity": "c_c29ab0ccbfbb", "metric": "certificate count", "value": 680, "unit": None, "period": "2026", "qualifiers": {"rank": 4}, "required": True},
        {"claim_id": "Q6_E5", "text": "C_534b9225f01a has 430 certificates with 0.2% on-time rate", "entity": "c_534b9225f01a", "metric": "certificate count", "value": 430, "unit": None, "period": "2026", "qualifiers": {"rank": 5}, "required": True},
    ],
    "Q7": [
        {"claim_id": "Q7_E1", "text": "January 2026 on-time rate is 55.7%", "entity": None, "metric": "on-time completion rate", "value": 55.7, "unit": "%", "period": "january 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q7_E2", "text": "February 2026 on-time rate is 56.8%", "entity": None, "metric": "on-time completion rate", "value": 56.8, "unit": "%", "period": "february 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q7_E3", "text": "March 2026 on-time rate is 71.0% and is the best month", "entity": None, "metric": "on-time completion rate", "value": 71.0, "unit": "%", "period": "march 2026", "qualifiers": {"superlative": "best"}, "required": True},
        {"claim_id": "Q7_E4", "text": "April 2026 on-time rate is 67.1%", "entity": None, "metric": "on-time completion rate", "value": 67.1, "unit": "%", "period": "april 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q7_E5", "text": "May 2026 on-time rate is 70.9%", "entity": None, "metric": "on-time completion rate", "value": 70.9, "unit": "%", "period": "may 2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q7_E6", "text": "June 2026 on-time rate is 68.4%", "entity": None, "metric": "on-time completion rate", "value": 68.4, "unit": "%", "period": "june 2026", "qualifiers": {}, "required": True},
    ],
    "Q8": [
        {"claim_id": "Q8_E1", "text": "ACTIVE companies have 65.0% on-time completion rate in 2026", "entity": "active", "metric": "on-time completion rate", "value": 65.0, "unit": "%", "period": "2026", "qualifiers": {"status": "ACTIVE"}, "required": True},
        {"claim_id": "Q8_E2", "text": "INACTIVE companies have 90.7% on-time completion rate in 2026", "entity": "inactive", "metric": "on-time completion rate", "value": 90.7, "unit": "%", "period": "2026", "qualifiers": {"status": "INACTIVE"}, "required": True},
        {"claim_id": "Q8_E3", "text": "ACTIVE companies have 12013 total completions with deadlines", "entity": "active", "metric": "total completions", "value": 12013, "unit": None, "period": "2026", "qualifiers": {"status": "ACTIVE"}, "required": True},
        {"claim_id": "Q8_E4", "text": "INACTIVE companies have only 194 total completions with deadlines", "entity": "inactive", "metric": "total completions", "value": 194, "unit": None, "period": "2026", "qualifiers": {"status": "INACTIVE"}, "required": True},
    ],
    "Q9": [
        {"claim_id": "Q9_E1", "text": "30 companies experienced 50%+ drops at least twice in 2026", "entity": None, "metric": "company count", "value": 30, "unit": None, "period": "2026", "qualifiers": {"condition": "50%+ drop twice"}, "required": True},
        {"claim_id": "Q9_E2", "text": "Combined certificate volume of affected companies is 6857", "entity": None, "metric": "certificate count", "value": 6857, "unit": None, "period": "2026", "qualifiers": {}, "required": True},
    ],
    "Q10": [
        {"claim_id": "Q10_E1", "text": "March cohort showed strongest engagement with 85.9 avg certificates per company", "entity": "march cohort", "metric": "average certificates per company", "value": 85.9, "unit": None, "period": "2026", "qualifiers": {"superlative": "strongest"}, "required": True},
        {"claim_id": "Q10_E2", "text": "January cohort has 5 companies producing 399 certificates", "entity": "january cohort", "metric": "certificate count", "value": 399, "unit": None, "period": "2026", "qualifiers": {"cohort_size": 5}, "required": True},
        {"claim_id": "Q10_E3", "text": "March cohort has 8 companies producing 687 certificates", "entity": "march cohort", "metric": "certificate count", "value": 687, "unit": None, "period": "2026", "qualifiers": {"cohort_size": 8}, "required": True},
    ],
    # --- Q11-Q25 Claims (extended set) ---
    "Q11": [
        {"claim_id": "Q11_E1", "text": "186 companies are currently in ACTIVE status", "entity": None, "metric": "company count", "value": 186, "unit": None, "period": None, "qualifiers": {"status": "ACTIVE"}, "required": True},
        {"claim_id": "Q11_E2", "text": "Total number of companies is 1217", "entity": None, "metric": "company count", "value": 1217, "unit": None, "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q11_E3", "text": "Active companies represent approximately 15.3% of total", "entity": None, "metric": "share", "value": 15.3, "unit": "%", "period": None, "qualifiers": {"status": "ACTIVE"}, "required": True},
    ],
    "Q13": [
        {"claim_id": "Q13_E1", "text": "476 distinct Wahlkurs (elective) courses exist", "entity": None, "metric": "course count", "value": 476, "unit": None, "period": None, "qualifiers": {"course_type": "Wahlkurs"}, "required": True},
        {"claim_id": "Q13_E2", "text": "421 distinct Pflichtkurs (mandatory) courses exist", "entity": None, "metric": "course count", "value": 421, "unit": None, "period": None, "qualifiers": {"course_type": "Pflichtkurs"}, "required": True},
        {"claim_id": "Q13_E3", "text": "Total of approximately 897 distinct courses across both types", "entity": None, "metric": "course count", "value": 897, "unit": None, "period": None, "qualifiers": {}, "required": True},
    ],
    "Q14": [
        {"claim_id": "Q14_E1", "text": "The newest companies in 2026 were created in July 2026", "entity": None, "metric": None, "value": None, "unit": None, "period": "july 2026", "qualifiers": {"superlative": "newest"}, "required": True},
        {"claim_id": "Q14_E2", "text": "Most of the newest companies have TRIAL status", "entity": None, "metric": None, "value": None, "unit": None, "period": "2026", "qualifiers": {"status": "TRIAL"}, "required": True},
    ],
    "Q16": [
        {"claim_id": "Q16_E1", "text": "3322 total assignments exist", "entity": None, "metric": "assignment count", "value": 3322, "unit": None, "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q16_E2", "text": "1370 assignments have a deadline", "entity": None, "metric": "assignment count", "value": 1370, "unit": None, "period": None, "qualifiers": {"has_deadline": True}, "required": True},
        {"claim_id": "Q16_E3", "text": "1952 assignments have no deadline", "entity": None, "metric": "assignment count", "value": 1952, "unit": None, "period": None, "qualifiers": {"has_deadline": False}, "required": True},
        {"claim_id": "Q16_E4", "text": "75 assignments were completed late", "entity": None, "metric": "late count", "value": 75, "unit": None, "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q16_E5", "text": "Approximately 5.5% of assignments with deadlines were completed late", "entity": None, "metric": "late rate", "value": 5.5, "unit": "%", "period": None, "qualifiers": {}, "required": True},
    ],
    "Q22": [
        {"claim_id": "Q22_E1", "text": "815 companies have assignments but zero certificates in 2026", "entity": None, "metric": "company count", "value": 815, "unit": None, "period": "2026", "qualifiers": {"pattern": "non-converting"}, "required": True},
        {"claim_id": "Q22_E2", "text": "937 total companies have assignments", "entity": None, "metric": "company count", "value": 937, "unit": None, "period": None, "qualifiers": {"has_assignments": True}, "required": True},
        {"claim_id": "Q22_E3", "text": "Only 122 companies with assignments earned certificates in 2026", "entity": None, "metric": "company count", "value": 122, "unit": None, "period": "2026", "qualifiers": {"has_certificates": True}, "required": True},
    ],
    "Q23": [
        {"claim_id": "Q23_E1", "text": "248 users earned certificates in at least 3 consecutive months in 2026", "entity": None, "metric": "user count", "value": 248, "unit": None, "period": "2026", "qualifiers": {"consecutive_months": 3}, "required": True},
        {"claim_id": "Q23_E2", "text": "1447 total users earned certificates in 2026", "entity": None, "metric": "user count", "value": 1447, "unit": None, "period": "2026", "qualifiers": {}, "required": True},
        {"claim_id": "Q23_E3", "text": "Consistently active users represent approximately 17.1% of total", "entity": None, "metric": "share", "value": 17.1, "unit": "%", "period": "2026", "qualifiers": {}, "required": True},
    ],
    "Q25": [
        {"claim_id": "Q25_E1", "text": "Overall exam failure rate is approximately 35.1%", "entity": None, "metric": "failure rate", "value": 35.1, "unit": "%", "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q25_E2", "text": "56527 total exams in the dataset", "entity": None, "metric": "exam count", "value": 56527, "unit": None, "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q25_E3", "text": "19822 exams were failed", "entity": None, "metric": "exam count", "value": 19822, "unit": None, "period": None, "qualifiers": {"outcome": "failed"}, "required": True},
    ],
    # Q12, Q15, Q18, Q21, Q24 are UNANSWERABLE — evaluated via abstention_correctness, no claims needed
    "Q17": [
        {"claim_id": "Q17_E1", "text": "Baercare companies assign an average of 5.6 distinct courses per company", "entity": "baercare", "metric": "avg distinct courses per company", "value": 5.6, "unit": None, "period": None, "qualifiers": {"superlative": "most diverse"}, "required": True},
        {"claim_id": "Q17_E2", "text": "Care companies assign an average of 4.5 distinct courses per company", "entity": "care", "metric": "avg distinct courses per company", "value": 4.5, "unit": None, "period": None, "qualifiers": {}, "required": True},
        {"claim_id": "Q17_E3", "text": "Baercare has the most diverse course portfolio with 256 distinct courses across 46 companies", "entity": "baercare", "metric": "distinct course count", "value": 256, "unit": None, "period": None, "qualifiers": {"num_companies": 46}, "required": True},
        {"claim_id": "Q17_E4", "text": "Care has 68 distinct courses across 15 companies", "entity": "care", "metric": "distinct course count", "value": 68, "unit": None, "period": None, "qualifiers": {"num_companies": 15}, "required": True},
    ],
    "Q19": [
        {"claim_id": "Q19_E1", "text": "C_fcc053f19146 has the highest conversion rate at 415.3 (1246 certs / 3 assignments)", "entity": "c_fcc053f19146", "metric": "conversion rate", "value": 415.333, "unit": None, "period": "2026", "qualifiers": {"rank": 1}, "required": True},
        {"claim_id": "Q19_E2", "text": "C_33a29351021c has conversion rate 109.0 (109 certs / 1 assignment)", "entity": "c_33a29351021c", "metric": "conversion rate", "value": 109.0, "unit": None, "period": "2026", "qualifiers": {"rank": 2}, "required": True},
        {"claim_id": "Q19_E3", "text": "C_c0238f1b43b5 has conversion rate 107.5 (215 certs / 2 assignments)", "entity": "c_c0238f1b43b5", "metric": "conversion rate", "value": 107.5, "unit": None, "period": "2026", "qualifiers": {"rank": 3}, "required": True},
        {"claim_id": "Q19_E4", "text": "C_fb60520cf036 has conversion rate 101.0 (101 certs / 1 assignment)", "entity": "c_fb60520cf036", "metric": "conversion rate", "value": 101.0, "unit": None, "period": "2026", "qualifiers": {"rank": 4}, "required": True},
        {"claim_id": "Q19_E5", "text": "C_b659e12972b7 has conversion rate 84.7 (1016 certs / 12 assignments)", "entity": "c_b659e12972b7", "metric": "conversion rate", "value": 84.667, "unit": None, "period": "2026", "qualifiers": {"rank": 5}, "required": True},
        {"claim_id": "Q19_E6", "text": "122 companies have both certificates and assignments in 2026", "entity": None, "metric": "company count", "value": 122, "unit": None, "period": "2026", "qualifiers": {"has_both": True}, "required": True},
    ],
    "Q20": [
        {"claim_id": "Q20_E1", "text": "Mandatory courses (Pflichtkurs) average -3.4 days relative to deadline (completed 3.4 days before deadline)", "entity": None, "metric": "avg days vs deadline", "value": -3.4, "unit": "days", "period": None, "qualifiers": {"course_type": "Pflichtkurs"}, "required": True},
        {"claim_id": "Q20_E2", "text": "Only mandatory courses appear in completed assignments with deadlines (no elective data)", "entity": None, "metric": None, "value": None, "unit": None, "period": None, "qualifiers": {"data_limitation": "no elective completions with deadlines"}, "required": True},
        {"claim_id": "Q20_E3", "text": "Care companies average -217.7 days vs deadline (completed much earlier)", "entity": "care", "metric": "avg days vs deadline", "value": -217.7, "unit": "days", "period": None, "qualifiers": {"n_completed": 15}, "required": True},
        {"claim_id": "Q20_E4", "text": "Baercare companies average -18.2 days vs deadline", "entity": "baercare", "metric": "avg days vs deadline", "value": -18.2, "unit": "days", "period": None, "qualifiers": {"n_completed": 87}, "required": True},
        {"claim_id": "Q20_E5", "text": "144 assignments have both completion and deadline dates with 47.9% on-time rate", "entity": None, "metric": "on-time rate", "value": 47.9, "unit": "%", "period": None, "qualifiers": {"total_with_dates": 144}, "required": True},
    ],
    # Q12, Q15, Q18, Q21, Q24 are UNANSWERABLE — evaluated via abstention_correctness, no claims needed
    # NOTE: Q7 is defined in the main Q1-Q10 block above (correct values: 55.7, 56.8, 71.0, 67.1, 70.9, 68.4)
    # A duplicate Q7 with wrong values (53.6, 48.4, 65.2...) was removed here to prevent dict-key overwrite.
}

# --- Merge expected_claims into EVALUATION_QUESTIONS ---
for q in EVALUATION_QUESTIONS:
    qid = q["id"]
    if qid in _EXPECTED_CLAIMS:
        q["expected_claims"] = _EXPECTED_CLAIMS[qid]
    else:
        # Q6, Q8, Q9, Q10 have qualitative answers — use prepare_expected_claims() to generate
        q.setdefault("expected_claims", [])

_with_claims = sum(1 for q in EVALUATION_QUESTIONS if q.get("expected_claims"))
print(f"✓ Expected claims loaded: {_with_claims}/{len(EVALUATION_QUESTIONS)} questions have ground-truth claims")
print(f"  Q6, Q8, Q9, Q10 require prepare_expected_claims() or manual review")